## Reproducing L96 data generation 

This notebook can be used to produce the Lorenz 96 datasets for both the reference (two-scale system) and the emulator (single scale system). The construction of the dataset sub-trajectories $\mathrm{dim}(\mathbb{D})$ is based on fractions of the chaotic decorrelation of the system.

The L96 "two-timescales" system is given by two coupled ODEs:

\begin{align}
\frac{d}{dt} X_k &= -X_{k-1} (X_{k-2} - X_{k+1}) - X_k + S - \left(\frac{hc}{b}\right) \sum_{j = 1}^{N_{j}} Y_{j,k} \\
\frac{d}{dt} Y_{j,k} &= -cbY_{j+1,k} (Y_{j+2,k} - Y_{j-1,k}) - cY_{j,k} + \frac{hc}{b} X_k
\end{align}

To reproduce the paper's experiments, we take: 
* $b = c = 10$
* $h = 1$
* $S = 18$
* $N_k = 8$
* $N_{j} = 20$

Note that $X_k$ is 1-D, while $Y_{j,k}$ is 2-D.

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import tqdm
import h5py
import matplotlib.pyplot as plt

plt.rcParams.update({
  'mathtext.fontset': 'cm'
})

import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jnr

jax.config.update(
  'jax_enable_x64', True
)

import models.time_solver as stepper
from models.l96 import (
    L96, 
    dynamical_solver,
    dynamical_solver_single
)


### Initializing a new two-timescales simulation

We create a new dynamical solver with the configuration parameters and create random initial conditions $X_k \sim \mathcal{N}(0, b^2)$ and $Y_{j,k} \sim \mathcal{N}(0, 1)$. At this resolution, $\delta t = 10^{-3}$ ensures stable behavior with the RK4 integrator.


In [ ]:
cfg_name = 'l96'
data_path = os.path.join(os.path.join(os.getcwd(), '../data'), cfg_name)

eq = L96(
    b=10.0,
    c=10.0,
    h=1.0,
    n_k=8,
    n_j=20
)
print(eq)

source_val = 18.0
def source_term(x_k, t):
    return source_val

key = jnr.key(42)
key, x_k_key, y_j_key = jnr.split(key, 3)

# Random initial conditions
x_k = eq.b * jnr.normal(x_k_key, eq.n_k)
y_j = jnr.normal(y_j_key, (eq.n_j, eq.n_k))

dt = 0.001
solver = jax.jit(dynamical_solver(
    eq,
    stepper.RK4(dt),
    source_term
))

### Computing the decorrelation time (RMSE-based)

From random initial conditions $X_k, Y_{j,k}$ and a perturbed state $\delta X_k = X_k + 10^{-12} P_k$ where $P_k \sim \mathcal{N}(0, 1)$, run an ensemble of 25 perturbed simulations.

In [ ]:
time = 0.0
iters = 10000
logs = 500
logs_freq = int(iters / logs)

ens_members = 25

time_t = []
x_k_t = []
y_j_t = []

x_k_0 = np.copy(x_k)
y_j_0 = np.copy(y_j)

print('Computing reference trajectory...')
pbar = tqdm.tqdm(range(iters), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for i in pbar:
    x_k, y_j = solver(x_k, y_j, time)
    time += dt
    if i % logs_freq == 0:
        time_t.append(time)

        x_k_t.append(
            x_k
        )
        
        y_j_t.append(
            y_j
        )

ens_rmse_t = np.zeros((ens_members, logs))

print('Computing perturbed trajectory for ensemble members...')
pbar = tqdm.tqdm(range(ens_members), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for e in pbar:
    key, pert_key = jnr.split(key)
    x_k_p = x_k_0 + 1e-12 * jnr.normal(pert_key, eq.n_k)
    y_j_p = y_j_0.copy()
    for i in range(iters):
        x_k_p, y_j_p = solver(x_k_p, y_j_p, time)
        time += dt
        if i % logs_freq == 0:
            ens_rmse_t[e, i // logs_freq] = np.sqrt(np.mean(np.square(x_k_p - x_k_t[i // logs_freq])))

fig, axs = plt.subplots(ncols=1, nrows=1, figsize=(2.5, 3.5), dpi=120)

axs.semilogy(time_t, ens_rmse_t.T, alpha=0.3)
axs.semilogy(time_t, np.mean(ens_rmse_t, axis=0), color='k')
axs.axvline(6, linestyle='--', color='k')
axs.set_xlabel(r'$t$', fontsize=15)
axs.set_ylabel(r'$\langle (\delta X_{k}(t) - X_{k}(t))^{2} \rangle^{1/2}$', fontsize=15)
axs.tick_params(reset=True, axis='both', which='both', direction='in')

fig.tight_layout()
fig.savefig(os.path.join(data_path, 'decorr.pdf'))
plt.show()

In [ ]:
def get_dataset_stats(
    eq: L96, 
    dt: float,
    decorr_t: float,
    frac_decorr: float,
):
    n_decorr_iters = int(decorr_t / dt)
    print('Iterations required for full decorrelation =',n_decorr_iters)

    n_steps = int(frac_decorr * n_decorr_iters / eq.n_j)
    print('Number of steps in sub-trajectories =',n_steps)
    return n_steps

n_steps = get_dataset_stats(
    eq,
    dt=dt,
    decorr_t=6.0, # Full decorrelation at t ~ 6
    frac_decorr=0.1, # 10%
)

### Generating a dataset of 25 trajectories from the two-timescales reference:

In [ ]:
n_trajs = 25

time = 0.0
iters = n_trajs * n_steps * eq.n_j

time_t = []
x_k_t = []
y_j_t = []

pbar = tqdm.tqdm(range(iters), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for i in pbar:
    x_k, y_j = solver(x_k, y_j, time)
    time += dt

    if i % eq.n_j == 0:
        time_t.append(time)
        
        x_k_t.append(
            x_k
        )
        y_j_t.append(
            y_j
        )
    
x_k_ref = np.array(np.split(np.array(x_k_t), n_trajs, axis=0))
y_j_ref = np.array(np.split(np.array(y_j_t), n_trajs, axis=0))
time_ref = np.array(np.split(np.array(time_t), n_trajs, axis=0))
print('Training reference (DNS) dataset shape =',x_k_ref.shape)

### Every 30 steps, restart and run the single-timescale solver for the neural emulation dataset:

In [ ]:
# "Single-timescale" model, with N_j = 0
eq_single = L96(
    b=eq.b,
    c=eq.c,
    h=eq.h,
    n_k=eq.n_k,
    n_j=0
)

dt_single = dt * eq.n_j

# Baseline "single-timescale" without correction
solver_single = jax.jit(dynamical_solver_single(
    eq_single,
    stepper.RK4(dt_single),
    source_term
))

x_k_emu = np.zeros_like(x_k_ref)
pbar = tqdm.tqdm(range(n_trajs), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for traj in pbar:
    x_k_emu[traj, 0] = x_k_ref[traj, 0]
    time = time_ref[traj]
    for step in range(1, n_steps):
        x_k_emu[traj, step], _ = solver_single(x_k_emu[traj, step - 1], 0, time)
        time += dt_single

fig, axs = plt.subplots(ncols=1, nrows=1, figsize=(7.5, 3.5), dpi=120)

axs.plot(time_t, np.mean(x_k_ref, axis=-1).flatten(), label='Two-timescale reference')
axs.plot(time_t, np.mean(x_k_emu, axis=-1).flatten(), '-o', markevery=n_steps, label='Single timescale (restarted) trajectories')
axs.set_xlabel(r'$t$', fontsize=15)
axs.set_ylabel(r'$\langle X_k \rangle_k(t)$', fontsize=15)
axs.tick_params(reset=True, axis='both', which='both', direction='in')
axs.legend(fontsize=9, frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
if not os.path.exists(data_path):
    os.makedirs(data_path)
print('Saving datasets...')
with h5py.File(os.path.join(data_path, 'datasets.h5'), 'w') as f:
    f.attrs['dt'] = dt
    f.attrs['n_steps'] = n_steps
    f.attrs['n_trajs'] = n_trajs
    f.attrs['source_val'] = source_val

    f.create_dataset('x_k_ref',
                     data=np.array(x_k_ref))
    f.create_dataset('y_j_ref',
                     data=np.array(y_j_ref))
    f.create_dataset('x_k_emu',
                     data=np.array(x_k_emu))
eq.save(
    os.path.join(data_path, 'snapshot.h5'),
    0.0,
    x_k, 
    y_j,
)